Частина 2

In [31]:
import pandas as pd
import numpy as np
import timeit

Завдання 1
Завантажити та відкрити датасет Individual Household Electric Power Consumption Dataset.

In [32]:
#завантаження датасету
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    low_memory=False
)

print("Розмір датасету:", df.shape)
df.head()

Розмір датасету: (2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0



Завдання 2
Здійснити data cleaning

In [33]:
#замінюємо ? на NaN
df.replace("?", np.nan, inplace=True)

#видаляємо пропуски
df.dropna(inplace=True)

#перетворюємо числові колонки
numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

df[numeric_cols] = df[numeric_cols].astype(float)

print("Після очищення:", df.shape)

Після очищення: (2049280, 9)



Завдання 3
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [34]:
def power_over_5kw(df):

    result = df[df["Global_active_power"] > 5]

    print("Кількість записів:", len(result))

    return result
power_over_5kw(df)

Кількість записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0
...,...,...,...,...,...,...,...,...,...
2069356,22/11/2010,18:40:00,5.408,0.150,231.50,23.6,48.0,0.0,0.0
2069357,22/11/2010,18:41:00,5.528,0.144,232.48,24.6,53.0,0.0,0.0
2071586,24/11/2010,07:50:00,5.172,0.050,235.18,22.0,0.0,38.0,17.0
2071587,24/11/2010,07:51:00,5.750,0.000,234.40,24.6,0.0,39.0,17.0


Завдання 4
Обрати всі записи, у яких сила струму лежить в межах 19–20 А. Для них визначити ті, у яких пральна машина та холодильник споживають більше електроенергії ніж бойлер та кондиціонер.

In [35]:
#функція відбору сила струму 19 20 ампер
def current_range(df):

    subset = df[
        (df["Global_intensity"] >= 19) &
        (df["Global_intensity"] <= 20)
    ]

    result = subset[
        subset["Sub_metering_2"] >
        (subset["Sub_metering_1"] + subset["Sub_metering_3"])
    ]

    return result

result2 = current_range(df)

print("Кількість записів:", result2.shape[0])
result2.head()

Кількість записів: 2153


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


Завдання 5
Обрати випадковим чином 500000 записів без повторів та обчислити середні значення трьох груп споживання електроенергії.

In [36]:
#функція випадкової вибірки 500000 записів
def random_sample_mean(df):

    sample = df.sample(n=500000, replace=False)

    means = sample[
        ["Sub_metering_1",
         "Sub_metering_2",
         "Sub_metering_3"]
    ].mean()

    return means

result3 = random_sample_mean(df)

print("Середні значення:")
print(result3)

Середні значення:
Sub_metering_1    1.122910
Sub_metering_2    1.310630
Sub_metering_3    6.455984
dtype: float64


Завдання 6
Обрати записи після 18:00 із середнім споживанням понад 6 кВт, визначити ті, де група 2 має найбільше споживання. Після цього обрати кожен третій результат із першої половини та кожен четвертий із другої половини.

In [37]:
df["Time"] = pd.to_datetime(df["Time"])
df["Hour"] = df["Time"].dt.hour
#функція відбору після 18 години
def evening_filter(df):

    subset = df[
        (df["Hour"] >= 18) &
        (df["Global_active_power"] > 6)
    ]

    # група 2 найбільша
    subset = subset[
        (subset["Sub_metering_2"] >
         subset["Sub_metering_1"]) &
        (subset["Sub_metering_2"] >
         subset["Sub_metering_3"])
    ]

    half = len(subset) // 2

    first_half = subset.iloc[:half]
    second_half = subset.iloc[half:]

    result = pd.concat([
        first_half.iloc[::3],
        second_half.iloc[::4]
    ])

    return result

result4 = evening_filter(df)

print("Кількість відібраних записів:", result4.shape[0])
result4.head()

C:\Users\admin\AppData\Local\Temp\ipykernel_22448\1369998742.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Time"] = pd.to_datetime(df["Time"])


Кількість відібраних записів: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Hour
41,16/12/2006,2026-03-15 18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,18
44,16/12/2006,2026-03-15 18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,18
17494,28/12/2006,2026-03-15 20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,20
17498,28/12/2006,2026-03-15 21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,21
17501,28/12/2006,2026-03-15 21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,21


Завдання 7
Пронормувати вибраний датасет.

In [38]:
def normalize(df):
    return (df - df.min()) / (df.max() - df.min())

normalized = normalize(df[numeric_cols])
normalized.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


Завдання 8
Стандартизувати датасет.

In [39]:
def standardize(df):
    return (df - df.mean()) / df.std()

standardized = standardize(df[numeric_cols])
standardized.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955076,2.610720,-1.851816,3.098788,-0.182337,-0.051274,1.249420
1,4.037084,2.770405,-2.225274,4.133799,-0.182337,-0.051274,1.130897
2,4.050325,3.320431,-2.330213,4.133799,-0.182337,0.120487,1.249420
3,4.063566,3.355916,-2.191323,4.133799,-0.182337,-0.051274,1.249420
4,2.434881,3.586572,-1.592555,2.513781,-0.182337,-0.051274,1.249420


Завдання 9
Підрахувати коефіцієнти Пірсона та Спірмена.

In [40]:
#обчислення коефіцієнта спірмена
pearson = df["Global_active_power"].corr(
    df["Global_intensity"],
    method="pearson"
)

spearman = df["Global_active_power"].corr(
    df["Global_intensity"],
    method="spearman"
)

print("Pearson:", pearson)
print("Spearman:", spearman)

Pearson: 0.9988886002095828
Spearman: 0.9953719367299748



Завдання 10
Провести One Hot Encoding категоріального атрибута.

In [41]:
#one hot encoding для години
encoded = pd.get_dummies(df["Hour"], prefix="Hour")

encoded.head()

,Hour_0,Hour_1,Hour_2,Hour_3,Hour_4,Hour_5,Hour_6,Hour_7,Hour_8,Hour_9,...,Hour_14,Hour_15,Hour_16,Hour_17,Hour_18,Hour_19,Hour_20,Hour_21,Hour_22,Hour_23
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
